# Assignment 1 — NumPy: vectorised computation
No Python `for` loop over array elements is used anywhere, with one exception: Q8, where the question itself asks for a plain Python loop as the baseline. There it loops over a Python *list* (`arr.tolist()`), which the CONDITION allows.

In [1]:
import numpy as np
import pandas as pd
import time
from numpy.lib.stride_tricks import sliding_window_view

np.random.seed(42)

## Q1 — Create A, inspect dtype and memory

In [2]:
A = np.random.normal(loc=50, scale=12, size=(1000, 5))
print("shape :", A.shape)
print("dtype :", A.dtype)
print("bytes :", A.nbytes)

A32 = A.astype(np.float32)
print("bytes after float32:", A32.nbytes)

shape : (1000, 5)
dtype : float64
bytes : 40000
bytes after float32: 20000


## Q2 — Standardise every column (one broadcasting expression)

In [3]:
Z = (A - A.mean(axis=0)) / A.std(axis=0)
print("means:", np.round(Z.mean(axis=0), 6))
print("stds :", np.round(Z.std(axis=0), 6))

means: [-0. -0.  0. -0. -0.]
stds : [1. 1. 1. 1. 1.]


## Q3 — `minmax(X, axis)`

In [4]:
def minmax(X, axis):
    mn = X.min(axis=axis, keepdims=True)
    mx = X.max(axis=axis, keepdims=True)
    return (X - mn) / (mx - mn)

M0 = minmax(A, axis=0)
M1 = minmax(A, axis=1)
print("axis=0  min per column:", M0.min(axis=0))
print("axis=0  max per column:", M0.max(axis=0))
print("axis=1  min per row (first 5):", M1.min(axis=1)[:5])
print("axis=1  max per row (first 5):", M1.max(axis=1)[:5])
print("axis=1  all row mins 0 and maxes 1:", np.allclose(M1.min(axis=1), 0) and np.allclose(M1.max(axis=1), 1))

axis=0  min per column: [0. 0. 0. 0. 0.]
axis=0  max per column: [1. 1. 1. 1. 1.]
axis=1  min per row (first 5): [0. 0. 0. 0. 0.]
axis=1  max per row (first 5): [1. 1. 1. 1. 1.]
axis=1  all row mins 0 and maxes 1: True


## Q4 — Inject NaN into 5% of positions, fill with column means

In [5]:
n_nan = int(0.05 * A.size)                      # exactly 5% of 5000 = 250 positions
pos = np.random.choice(A.size, size=n_nan, replace=False)
A_nan = A.copy()
A_nan.flat[pos] = np.nan
print("NaNs injected:", np.isnan(A_nan).sum(), "of", A_nan.size)

col_means = np.nanmean(A_nan, axis=0)           # ignores NaN
mask = np.isnan(A_nan)
A = np.where(mask, col_means, A_nan)            # masked replacement with own-column mean

print("NaN left:", np.isnan(A).sum())
assert np.isnan(A).sum() == 0

NaNs injected: 250 of 5000
NaN left: 0


## Q5 — Replace values beyond 3 standard deviations with the column median

In [6]:
mu  = A.mean(axis=0)
sd  = A.std(axis=0)
med = np.median(A, axis=0)

outliers = np.abs(A - mu) > 3 * sd
print("values replaced per column:", outliers.sum(axis=0))
print("total replaced:", outliers.sum())

A = np.where(outliers, med, A)

values replaced per column: [5 1 5 3 5]
total replaced: 19


## Q6 — 200 × 200 Euclidean distance matrix (broadcasting only)

In [7]:
X = A[:200]                                     # (200, 5)
diff = X[:, None, :] - X[None, :, :]            # (200, 200, 5)
D = np.sqrt((diff ** 2).sum(axis=-1))           # (200, 200)

print("shape:", D.shape)
print("diagonal is all 0 :", np.allclose(np.diag(D), 0))
print("D equals D.T      :", np.allclose(D, D.T))

shape: (200, 200)
diagonal is all 0 : True
D equals D.T      : True


## Q7 — 5 nearest rows per row with `argpartition`

In [8]:
D_ex = D.copy()
np.fill_diagonal(D_ex, np.inf)                  # exclude the row itself

part = np.argpartition(D_ex, kth=5, axis=1)[:, :5]          # the 5 smallest, unordered
order = np.argsort(np.take_along_axis(D_ex, part, axis=1), axis=1)
nn = np.take_along_axis(part, order, axis=1)    # tidy: nearest first (sorts only 5 items per row)

print("shape:", nn.shape, "| dtype:", nn.dtype)
print("self never a neighbour:", not (nn == np.arange(200)[:, None]).any())
print(nn[:5])

shape: (200, 5) | dtype: int64
self never a neighbour: True
[[180 179 116 178 192]
 [114  17  31 112 197]
 [  3 158 132 145 102]
 [132   2 101   5 194]
 [160 158 143 102  43]]


**Why `argpartition`:** it finds the 5 smallest of 200 in O(n) per row, whereas a full sort costs O(n log n) and orders 195 entries we never use.

## Q8 — `np.bincount` vs a plain Python loop

In [9]:
arr = np.random.randint(0, 100, size=1_000_000)   # integers 0..99

In [10]:
%%time
_t = time.perf_counter()
counts_np = np.bincount(arr, minlength=100)
t_np = time.perf_counter() - _t

CPU times: user 2.31 ms, sys: 0 ns, total: 2.31 ms
Wall time: 2.32 ms


In [11]:
%%time
_t = time.perf_counter()
counts_py = [0] * 100
for v in arr.tolist():          # loop over a Python list, as the CONDITION permits
    counts_py[v] += 1
t_py = time.perf_counter() - _t

CPU times: user 55.9 ms, sys: 3 ms, total: 58.9 ms
Wall time: 62.4 ms


In [12]:
assert np.array_equal(counts_np, np.array(counts_py))
speedup = t_py / t_np
print(f"bincount: {t_np*1000:.2f} ms | Python loop: {t_py*1000:.2f} ms")
print(f"Speed-up: {speedup:.1f}x")

bincount: 2.30 ms | Python loop: 62.34 ms
Speed-up: 27.1x


## Q9 — Sliding windows of length 10

In [13]:
x = np.random.normal(size=10_000)
W = sliding_window_view(x, 10)
print("windows shape:", W.shape)

win_mean = W.mean(axis=1)
pd_mean = pd.Series(x).rolling(10).mean().to_numpy()[9:]   # drop the 9 leading NaNs
print("matches pandas rolling(10).mean():", np.allclose(win_mean, pd_mean))

windows shape: (9991, 10)
matches pandas rolling(10).mean(): True


## Q10 — Reshape (12, 8) → (4, 3, 8), mean over the middle axis

In [14]:
R = np.arange(96, dtype=float).reshape(12, 8)
R3 = R.reshape(4, 3, 8)
out = R3.mean(axis=1)
print("reshaped:", R3.shape, "| after mean over axis 1:", out.shape)
print(out)

reshaped: (4, 3, 8) | after mean over axis 1: (4, 8)
[[ 8.  9. 10. 11. 12. 13. 14. 15.]
 [32. 33. 34. 35. 36. 37. 38. 39.]
 [56. 57. 58. 59. 60. 61. 62. 63.]
 [80. 81. 82. 83. 84. 85. 86. 87.]]


**Meaning of the result `(4, 8)`:** the 12 rows were grouped into 4 blocks of 3 consecutive rows, and each block was averaged.

So we get one mean row per block, still across all 8 columns (e.g. 12 monthly rows of 8 features becoming 4 quarterly averages).

## Q11 — Save and reload

In [15]:
np.save("clean.npy", A)
loaded = np.load("clean.npy")
assert np.array_equal(A, loaded)
print("loaded array identical to original:", True)

loaded array identical to original: True
